Install required packages

In [1]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


Imports

In [2]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
import matplotlib.pyplot as plt
import seaborn as sns

Load datasets

In [3]:
train_df = pd.read_csv("train.csv")
validation_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2700, 2)
Validation shape: (338, 2)
Test shape: (338, 2)


Prepare text lists

In [4]:
train_questions = train_df["Question"].tolist()
train_answers = train_df["Answer"].tolist()

validation_questions = validation_df["Question"].tolist()
validation_answers = validation_df["Answer"].tolist()

test_questions = test_df["Question"].tolist()
test_answers = test_df["Answer"].tolist()

print("Training questions:", len(train_questions))
print("Validation questions:", len(validation_questions))
print("Test questions:", len(test_questions))

Training questions: 2700
Validation questions: 338
Test questions: 338


Load the fine-tuned MPNet model

In [5]:
finetuned_model = SentenceTransformer("mpnet_donor_finetuned")

print("Fine-tuned MPNet loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned MPNet loaded successfully.


Encode training questions for dense retrieval

In [6]:
finetuned_train_embeddings = finetuned_model.encode(
    train_questions,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Dense training embeddings created.")
print("Embedding shape:", finetuned_train_embeddings.shape)

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Dense training embeddings created.
Embedding shape: torch.Size([2700, 768])


Build BM25 corpus

In [7]:
def bm25_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+\- ]", " ", text)
    return text.split()

bm25_corpus = [bm25_tokenize(q) for q in train_questions]
bm25 = BM25Okapi(bm25_corpus)

print("BM25 corpus built successfully.")
print("Corpus size:", len(bm25_corpus))

BM25 corpus built successfully.
Corpus size: 2700


Define dense retrieval helper

In [8]:
def dense_retrieve(user_query, questions, answers, embeddings, model):
    query_embedding = model.encode(user_query, convert_to_tensor=True)
    similarities = util.cos_sim(query_embedding, embeddings)[0]
    
    best_idx = similarities.argmax().item()
    best_score = similarities[best_idx].item()
    
    return {
        "matched_question": questions[best_idx],
        "retrieved_answer": answers[best_idx],
        "dense_score": best_score,
        "best_idx": best_idx
    }

Define BM25 retrieval helper

In [9]:
def bm25_retrieve(user_query, questions, answers, bm25_model):
    tokenized_query = bm25_tokenize(user_query)
    scores = bm25_model.get_scores(tokenized_query)
    
    best_idx = int(np.argmax(scores))
    best_score = float(scores[best_idx])
    
    return {
        "matched_question": questions[best_idx],
        "retrieved_answer": answers[best_idx],
        "bm25_score": best_score,
        "best_idx": best_idx
    }

Define score normalization function

In [10]:
def minmax_normalize(scores):
    scores = np.array(scores, dtype=float)
    min_score = scores.min()
    max_score = scores.max()
    
    if max_score == min_score:
        return np.ones_like(scores)
    
    return (scores - min_score) / (max_score - min_score)

Define hybrid retrieval function

In [11]:
def hybrid_retrieve(user_query, questions, answers, embeddings, model, bm25_model, alpha=0.7):
    # Dense scores
    query_embedding = model.encode(user_query, convert_to_tensor=True)
    dense_scores = util.cos_sim(query_embedding, embeddings)[0].cpu().numpy()
    
    # BM25 scores
    tokenized_query = bm25_tokenize(user_query)
    bm25_scores = np.array(bm25_model.get_scores(tokenized_query), dtype=float)
    
    # Normalize both
    dense_norm = minmax_normalize(dense_scores)
    bm25_norm = minmax_normalize(bm25_scores)
    
    # Weighted hybrid score
    hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm
    
    best_idx = int(np.argmax(hybrid_scores))
    
    return {
        "matched_question": questions[best_idx],
        "retrieved_answer": answers[best_idx],
        "hybrid_score": float(hybrid_scores[best_idx]),
        "dense_score": float(dense_scores[best_idx]),
        "bm25_score": float(bm25_scores[best_idx]),
        "best_idx": best_idx
    }

Test one sample query

In [12]:
sample_result = hybrid_retrieve(
    user_query="Can I donate blood if I have malaria?",
    questions=train_questions,
    answers=train_answers,
    embeddings=finetuned_train_embeddings,
    model=finetuned_model,
    bm25_model=bm25,
    alpha=0.7
)

print("Matched Question:", sample_result["matched_question"])
print("Retrieved Answer:", sample_result["retrieved_answer"])
print("Hybrid Score:", round(sample_result["hybrid_score"], 4))
print("Dense Score:", round(sample_result["dense_score"], 4))
print("BM25 Score:", round(sample_result["bm25_score"], 4))

Matched Question: Can individuals who have traveled to countries with malaria risk donate blood, and if so, are there any waiting periods or restrictions?
Retrieved Answer: Depending on the destination, duration of travel, and risk of malaria exposure, individuals who have traveled to countries with malaria risk may be temporarily deferred from donating blood to prevent transmission risks. Donors should consult with blood donation centers for specific guidelines.
Hybrid Score: 1.0
Dense Score: 0.5618
BM25 Score: 20.1719


Run hybrid retrieval on validation set

In [13]:
hybrid_validation_results = []

for true_question, true_answer in zip(validation_questions, validation_answers):
    output = hybrid_retrieve(
        user_query=true_question,
        questions=train_questions,
        answers=train_answers,
        embeddings=finetuned_train_embeddings,
        model=finetuned_model,
        bm25_model=bm25,
        alpha=0.7
    )
    
    hybrid_validation_results.append({
        "validation_question": true_question,
        "expected_answer": true_answer,
        "matched_question": output["matched_question"],
        "retrieved_answer": output["retrieved_answer"],
        "hybrid_score": output["hybrid_score"],
        "dense_score": output["dense_score"],
        "bm25_score": output["bm25_score"]
    })

hybrid_validation_results_df = pd.DataFrame(hybrid_validation_results)

print("Hybrid validation retrieval completed.")
hybrid_validation_results_df.head()

Hybrid validation retrieval completed.


,validation_question,expected_answer,matched_question,retrieved_answer,hybrid_score,dense_score,bm25_score
0,What impact do intergenerational donation even...,"Intergenerational donation events, such as fam...",What impact do donor recognition programs and ...,Donor recognition programs and appreciation ev...,0.970588,0.933296,31.387055
1,How do blood drive hosts collaborate with acad...,Blood drive hosts collaborate with academic in...,How do blood drive hosts collaborate with tech...,Blood drive hosts collaborate with technology ...,0.965465,0.997448,29.149259
2,What factors contribute to disparities in bloo...,Disparities in blood donation rates among diff...,What factors contribute to disparities in bloo...,Disparities in blood donation rates among raci...,1.000000,0.919082,31.689027
3,What role can celebrity endorsements and partn...,Celebrity endorsements and influencer partners...,What role can celebrity endorsements and influ...,Celebrity endorsements and influencer partners...,0.993563,0.809935,43.859146
4,How does blood donation support families of pa...,Blood donation supports families of patients b...,How can donating blood promote intergeneration...,Blood donation can encourage families to parti...,0.923103,0.823986,11.694090


Compute semantic similarity of hybrid matches

In [14]:
hybrid_validation_similarity = []

for _, row in hybrid_validation_results_df.iterrows():
    q_emb = finetuned_model.encode(row["validation_question"], convert_to_tensor=True)
    m_emb = finetuned_model.encode(row["matched_question"], convert_to_tensor=True)
    sim = util.cos_sim(q_emb, m_emb).item()
    hybrid_validation_similarity.append(sim)

hybrid_validation_results_df["semantic_similarity"] = hybrid_validation_similarity

print(hybrid_validation_results_df["semantic_similarity"].describe())

count    338.000000
mean       0.923608
std        0.133923
min        0.296127
25%        0.917460
50%        0.989326
75%        0.998748
max        0.999951
Name: semantic_similarity, dtype: float64


Compute hybrid validation accuracy

In [15]:
hybrid_val_accuracy_085 = (hybrid_validation_results_df["semantic_similarity"] >= 0.85).mean() * 100
hybrid_val_accuracy_090 = (hybrid_validation_results_df["semantic_similarity"] >= 0.90).mean() * 100

print("Hybrid Validation Semantic Accuracy (>= 0.85):", round(hybrid_val_accuracy_085, 2), "%")
print("Hybrid Validation Strict Accuracy (>= 0.90):", round(hybrid_val_accuracy_090, 2), "%")

Hybrid Validation Semantic Accuracy (>= 0.85): 80.18 %
Hybrid Validation Strict Accuracy (>= 0.90): 76.04 %


Compare hybrid vs fine-tuned dense-only on validation

In [16]:
comparison_hybrid_val_df = pd.DataFrame({
    "Metric": [
        "Semantic Accuracy >= 0.85",
        "Strict Semantic Accuracy >= 0.90"
    ],
    "Fine-Tuned Dense Only": [
        83.14,
        77.81
    ],
    "Hybrid Retrieval": [
        round(hybrid_val_accuracy_085, 2),
        round(hybrid_val_accuracy_090, 2)
    ]
})

comparison_hybrid_val_df["Improvement"] = (
    comparison_hybrid_val_df["Hybrid Retrieval"] - comparison_hybrid_val_df["Fine-Tuned Dense Only"]
)

comparison_hybrid_val_df

,Metric,Fine-Tuned Dense Only,Hybrid Retrieval,Improvement
0,Semantic Accuracy >= 0.85,83.14,80.18,-2.96
1,Strict Semantic Accuracy >= 0.90,77.81,76.04,-1.77


Inspect weakest hybrid validation cases

In [17]:
worst_hybrid_val_cases = hybrid_validation_results_df.sort_values("semantic_similarity", ascending=True).head(20)

worst_hybrid_val_cases[[
    "validation_question",
    "matched_question",
    "retrieved_answer",
    "semantic_similarity",
    "dense_score",
    "bm25_score",
    "hybrid_score"
]]

,validation_question,matched_question,retrieved_answer,semantic_similarity,dense_score,bm25_score,hybrid_score
131,How can individuals benefit from quitting smok...,How can donating blood impact individuals with...,Donating blood could benefit individuals with ...,0.296127,0.296127,7.678456,0.780780
86,How can donating blood inspire acts of kindness?,How does donating blood promote greater empathy?,Donating blood promotes greater empathy by con...,0.309343,0.309343,9.468611,0.796836
79,How can individuals benefit from consuming mor...,How can individuals benefit from consuming mor...,Water-rich foods such as fruits and vegetables...,0.332128,0.332128,31.779793,0.870772
44,How can perceptions of blood donation impact d...,How do public perceptions of blood donation im...,"Public perceptions of blood donation, includin...",0.391295,0.391295,16.264753,0.909212
48,How do donor recruitment strategies differ bet...,How do donor recruitment strategies differ bet...,Donor recruitment strategies may differ betwee...,0.396620,0.396620,26.474023,0.809338
18,How can donating blood create lasting positive...,How can blood donation contribute to positive ...,Blood donation can contribute to positive ment...,0.408378,0.408378,11.204130,0.920366
28,What role does community mobilization play in ...,How does the program foster a sense of communi...,The program fosters a sense of community and b...,0.443708,0.443708,12.869018,0.874307
12,What role do donor education and pre-donation ...,What role does pre-donation counseling play in...,Pre-donation counseling plays a crucial role i...,0.459756,0.459756,27.799444,0.967405
166,How can donating blood impact individuals with...,How does donating blood contribute to better s...,Donating blood may contribute to better sleep ...,0.507890,0.507890,18.779549,0.915569
81,How can appointment scheduling systems be opti...,How can appointment scheduling systems incorpo...,Appointment scheduling systems can incorporate...,0.511124,0.511124,13.793369,0.856388


Save Phase 12 validation outputs

In [18]:
hybrid_validation_results_df.to_csv("phase12_hybrid_validation_results.csv", index=False)
comparison_hybrid_val_df.to_csv("phase12_hybrid_vs_dense_validation_comparison.csv", index=False)
worst_hybrid_val_cases.to_csv("phase12_hybrid_worst_validation_cases.csv", index=False)

print("Saved Phase 12 validation outputs.")

Saved Phase 12 validation outputs.


Create Phase 12 summary table

In [19]:
phase12_summary = {
    "Validation samples": len(hybrid_validation_results_df),
    "Hybrid Validation Accuracy (>= 0.85)": round(hybrid_val_accuracy_085, 2),
    "Hybrid Validation Accuracy (>= 0.90)": round(hybrid_val_accuracy_090, 2),
    "Average hybrid semantic similarity": round(hybrid_validation_results_df["semantic_similarity"].mean(), 4),
    "Minimum hybrid semantic similarity": round(hybrid_validation_results_df["semantic_similarity"].min(), 4),
    "Maximum hybrid semantic similarity": round(hybrid_validation_results_df["semantic_similarity"].max(), 4)
}

phase12_summary_df = pd.DataFrame(list(phase12_summary.items()), columns=["Metric", "Value"])
phase12_summary_df

,Metric,Value
0,Validation samples,338.0000
1,Hybrid Validation Accuracy (>= 0.85),80.1800
2,Hybrid Validation Accuracy (>= 0.90),76.0400
3,Average hybrid semantic similarity,0.9236
4,Minimum hybrid semantic similarity,0.2961
5,Maximum hybrid semantic similarity,1.0000
